## 1. Create a connection to the database using the sqlite3 library.

In [9]:
import sqlite3
import pandas as pd
conn = sqlite3.connect('../data/checking-logs.sqlite')

## 2. Get the schema of the test table.

In [10]:
query = "PRAGMA TABLE_INFO(test);"
test_schema = pd.read_sql_query(query, conn)
print(test_schema)

   cid             name       type  notnull dflt_value  pk
0    0              uid       TEXT        0       None   0
1    1          labname       TEXT        0       None   0
2    2  first_commit_ts  TIMESTAMP        0       None   0
3    3    first_view_ts  TIMESTAMP        0       None   0


## 3. Get only the first ten rows of the test table to see what it looks like.

In [11]:
test = pd.read_sql_query("SELECT * FROM test", conn)
test


,uid,labname,first_commit_ts,first_view_ts
0,user_1,laba04,2020-04-26 17:06:18.462708,2020-04-26 21:53:59.624136
1,user_1,laba04s,2020-04-26 17:12:11.843671,2020-04-26 21:53:59.624136
2,user_1,laba05,2020-05-02 19:15:18.540185,2020-04-26 21:53:59.624136
3,user_1,laba06,2020-05-17 16:26:35.268534,2020-04-26 21:53:59.624136
4,user_1,laba06s,2020-05-20 12:23:37.289724,2020-04-26 21:53:59.624136
5,user_1,project1,2020-05-14 20:56:08.898880,2020-04-26 21:53:59.624136
6,user_10,laba04,2020-04-25 08:24:52.696624,2020-04-18 12:19:50.182714
7,user_10,laba04s,2020-04-25 08:37:54.604222,2020-04-18 12:19:50.182714
8,user_10,laba05,2020-05-01 19:27:26.063245,2020-04-18 12:19:50.182714
9,user_10,laba06,2020-05-19 11:39:28.885637,2020-04-18 12:19:50.182714


## 4. Find the minimum value of the delta between the first commit and the deadline of the corresponding lab for all users using only one query.
Do this by joining the table with the deadlines table.
The difference should be displayed in hours.
Do not take lab project1 into account; it has longer deadlines and will be an outlier.
The value should be stored in the dataframe df_min with the corresponding uid.

In [12]:
query = "SELECT uid, MIN((deadlines - strftime('%s',first_commit_ts))/3600.0) as min_time FROM test t LEFT JOIN deadlines d ON t.labname = d.labs WHERE t.labname NOT LIKE 'project1';"
df_min = pd.read_sql_query(query, conn)
df_min

,uid,min_time
0,user_25,2.8675


## 5. Do the same thing for the maximum, but use only one query. The dataframe name is df_max.

In [13]:
query = "SELECT uid, MAX((deadlines - strftime('%s',first_commit_ts))/3600.0) as max_time FROM test t LEFT JOIN deadlines d ON t.labname = d.labs WHERE t.labname NOT LIKE 'project1' "
df_max = pd.read_sql_query(query, conn)
df_max

,uid,max_time
0,user_30,202.385


## 6. Do the same thing, but for the average. Use only one query. This time, your dataframe should not include the uid column. The dataframe name is df_avg.

In [14]:
query = "SELECT uid, AVG((deadlines - strftime('%s',first_commit_ts))/3600.0) as avg_time FROM test t LEFT JOIN deadlines d ON t.labname = d.labs WHERE t.labname NOT LIKE 'project1'"
df_avg = pd.read_sql_query(query, conn)
df_avg

,uid,avg_time
0,user_1,89.687841


## 7. We want to test the hypothesis that users who visited the newsfeed just a few times have a lower delta between the first commit and the deadline. To do this, calculate the correlation coefficient between the number of pageviews and the difference.
Using only one query, create a table with the following columns: "uid", "avg_diff", and "pageviews".
"uid" is the uids that exist in the test.
"avg_diff" is the average delta between the first commit and the lab deadline per user.
"pageviews" is the number of Newsfeed visits per user.
Do not take the lab project1 into account.
Store it in the dataframe views_diff.
Use the Pandas corr() method to calculate the correlation coefficient between the number of pageviews and the difference.

In [15]:
query = """
        SELECT 
        t.uid, 
        (AVG((deadlines - strftime('%s',first_commit_ts))/3600.0)) AS avg_diff, 
        pv.pageviews
        FROM test AS t 
        LEFT JOIN deadlines AS d
            ON t.labname = d.labs 
        LEFT JOIN (
            SELECT 
            uid,
            COUNT(*) AS pageviews
            FROM pageviews
            GROUP BY uid
        ) AS pv ON pv.uid = t.uid
        WHERE t.labname NOT LIKE 'project1' 
        GROUP BY t.uid;
        """

views_diff = pd.read_sql_query(query, conn)
corr = views_diff['pageviews'].corr(views_diff['avg_diff'])
print(corr)
views_diff

0.27914346425426556


,uid,avg_diff,pageviews
0,user_1,65.119778,28
1,user_10,75.242444,89
2,user_14,159.568796,143
3,user_17,62.207667,47
4,user_18,6.368148,3
5,user_19,99.440417,16
6,user_21,96.111181,10
7,user_25,93.474944,179
8,user_28,86.793833,149
9,user_3,105.738222,317


## 8. Close the connection.

In [16]:
conn.close()